# 00 — Construcción del dataset final

Une todo en dos archivos en `data/processed/`:
- **`panel.parquet`** (long, MultiIndex empresa×fecha) -> OE2 y OE5 (panel FE).
- **`series.parquet`** (wide, agregado) -> OE3 y OE4 (cartera + macro).

Maneja la ausencia de series del BCCh: si no están en `data/raw/`, usa solo FRED+precios
y avisa. Ejecutar DESPUÉS del notebook 01 (adquisición).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import build_panel as bp
from src import config as C

## 1. Elegir muestra (diseño de triangulación A/B/C)
- **B** `UNIVERSO_PRINCIPAL`: cobre puro, mercados globales (panel sólido).
- **A** `UNIVERSO_LOCAL_COBRE`: cobre puro, mercado chileno (Pucobre; serie de tiempo).
- **C** `UNIVERSO_LOCAL_MINERIA`: sector minero chileno (panel local).

Construye el panel para CADA muestra (cambia la variable, ajusta el sufijo de guardado y re-ejecuta).

In [ ]:
universo = C.UNIVERSO_PRINCIPAL          # muestra B (global, cobre puro)
# universo = C.UNIVERSO_LOCAL_COBRE      # muestra A (local, cobre puro: Pucobre)
# universo = C.UNIVERSO_LOCAL_MINERIA    # muestra C (local, sector minero)
list(universo.keys())

In [ ]:
panel, series = bp.construir(guardar=True, universo=universo)

In [ ]:
display(panel.head())
display(series.describe().T)

## 2. Chequeos de integridad
Antes de modelar: cobertura por empresa, % de faltantes, balance del panel.

In [ ]:
import pandas as pd
print('Observaciones por empresa:')
print(panel.groupby('empresa')['retorno'].count())
print('\n% faltantes por columna:')
print((panel.isna().mean()*100).round(1))

Listo. Ahora correr en orden: `02_eda_oe1` -> `03_oe2_panel` -> `04_oe3_cointegracion`
-> `05_oe4_var_irf` -> `06_oe5_fases_ciclo`.